In [ ]:
# Imports
import pandas as pd
import geopandas as gpd
import numpy as np
import h5py
from scipy.ndimage import label
import matplotlib.pyplot as plt
import os

In [ ]:
# Paths
fire_file_path = "../../Calabria_dataset/InputReteGood/Target/2017/20170701.h5"
climate_file_path = "../../Calabria_dataset/InputReteGood/Climatic/2017/20170701.h5"

In [ ]:
# Load cell-zone mapping
df_cell_zones = pd.read_parquet("../1_data/processed/cell_zones.parquet")

In [ ]:
# Load and parse fire H5 file
with h5py.File(fire_file_path, "r") as h5_file:
    fire_values_table = h5_file["values/table"][:]
    attributes_table = h5_file["attributes/table"][:]

index_values = fire_values_table["index"]
fire_results = fire_values_table["values_block_0"].flatten()

attribute_names = [attr[0].decode() for attr in attributes_table]
attribute_values = [attr[1][0] for attr in attributes_table]
attributes_dict = dict(zip(attribute_names, attribute_values))

ncols = int(attributes_dict["ncols"])
nrows = int(attributes_dict["nrows"])
cellsize = attributes_dict["cellsize"]
xllcorner = attributes_dict["xllcorner"]
yllcorner = attributes_dict["yllcorner"]

In [ ]:
# Convert index to row/column coordinates
row_coords = index_values // ncols
col_coords = index_values % ncols

In [ ]:
# Build 2D fire grid and label connected clusters
fire_grid = np.zeros((nrows, ncols), dtype=int)
for row, col, fire in zip(row_coords, col_coords, fire_results):
    if fire == 1:
        fire_grid[row, col] = 1

structure = np.array([[1, 1, 1],
                      [1, 1, 1],
                      [1, 1, 1]])
labeled_fire_grid, num_fires = label(fire_grid, structure=structure)

In [ ]:
# Extract fire cluster coordinates
fire_cluster_coords = []
for row in range(nrows):
    for col in range(ncols):
        cluster_id = labeled_fire_grid[row, col]
        if cluster_id > 0:
            x = xllcorner + (col * cellsize)
            y = yllcorner + ((nrows - 1 - row) * cellsize)
            fire_cluster_coords.append((cluster_id, row, col, x, y))

df_fire_clusters = pd.DataFrame(fire_cluster_coords,
    columns=["Cluster_ID", "Row", "Column", "X_Coord", "Y_Coord"])
df_fire_clusters["Y_Coord"] = df_fire_clusters["Y_Coord"].round(-0).astype(float).round(1)

In [ ]:
# Merge fire clusters with zone mapping
df_fire_clusters_zones = df_fire_clusters.merge(
    df_cell_zones[["X_Coord", "Y_Coord", "Zone_ID"]],
    on=["X_Coord", "Y_Coord"],
    how="left"
)

In [ ]:
# Assign dominant zone per cluster
df_dominant_zones = df_fire_clusters_zones.groupby("Cluster_ID")["Zone_ID"] \
    .agg(lambda x: x.mode().iloc[0]) \
    .reset_index() \
    .rename(columns={"Zone_ID": "Dominant_Zone_ID"})

df_fire_clusters_zones = df_fire_clusters_zones.drop(columns=["Zone_ID"]).merge(
    df_dominant_zones, on="Cluster_ID", how="left"
)
df_fire_clusters_zones.rename(columns={"Dominant_Zone_ID": "Zone_ID"}, inplace=True)

In [ ]:
# Count fire clusters per zone across all 8 zones
all_zones = pd.DataFrame(df_cell_zones["Zone_ID"].unique(), columns=["Zone_ID"])
df_fire_counts = df_fire_clusters_zones[["Cluster_ID", "Zone_ID"]].drop_duplicates()
fires_per_zone = df_fire_counts.groupby("Zone_ID").size().reset_index(name="Num_Fires")

df_zone_fire_counts = all_zones.merge(fires_per_zone, on="Zone_ID", how="left")
df_zone_fire_counts["Num_Fires"] = df_zone_fire_counts["Num_Fires"].fillna(0).astype(int)
df_zone_fire_counts = df_zone_fire_counts.sort_values("Zone_ID").reset_index(drop=True)

In [ ]:
# Load and parse climate H5 file
with h5py.File(climate_file_path, "r") as h5_file:
    values_table = h5_file["values/table"][:]
    attributes_table = h5_file["attributes/table"][:]

attribute_names = [attr[0].decode() for attr in attributes_table]
attribute_values = [attr[1][0] for attr in attributes_table]
attributes_dict = dict(zip(attribute_names, attribute_values))

ncols = int(attributes_dict["ncols"])
nrows = int(attributes_dict["nrows"])
cellsize = attributes_dict["cellsize"]

index_values = values_table["index"]
climate_values = values_table["values_block_0"]
row_coords = index_values // ncols
col_coords = index_values % ncols

In [ ]:
# Build climate DataFrame
df_climate = pd.DataFrame({
    "Row": row_coords,
    "Column": col_coords,
    "Precipitation": climate_values[:, 0],
    "Humidity": climate_values[:, 1],
    "Temperature": climate_values[:, 2],
    "Wind": climate_values[:, 3],
    "X_Coord": attributes_dict["xllcorner"] + (col_coords * cellsize),
    "Y_Coord": attributes_dict["yllcorner"] + ((nrows - 1 - row_coords) * cellsize)
})
df_climate["Y_Coord"] = df_climate["Y_Coord"].round(-0).astype(float).round(1)

In [ ]:
# Merge climate with zone mapping
df_climate_zone = df_cell_zones[["X_Coord", "Y_Coord", "Zone_ID"]].merge(
    df_climate, on=["X_Coord", "Y_Coord"], how="inner"
)

In [ ]:
# Compute zone-level average climate for all cells
df_zone_climate_all = df_climate_zone.groupby("Zone_ID").agg({
    "Precipitation": "mean",
    "Humidity": "mean",
    "Temperature": "mean",
    "Wind": "mean"
}).reset_index().rename(columns={
    "Precipitation": "Precipitation_all",
    "Humidity": "Humidity_all",
    "Temperature": "Temperature_all",
    "Wind": "Wind_all"
})

In [ ]:
# Compute zone-level average climate for fire cluster cells only
df_fire_with_climate = df_fire_clusters_zones.merge(
    df_climate[["X_Coord", "Y_Coord", "Precipitation", "Humidity", "Temperature", "Wind"]],
    on=["X_Coord", "Y_Coord"], how="left"
)

df_zone_fire_climate = df_fire_with_climate.groupby("Zone_ID").agg({
    "Precipitation": "mean",
    "Humidity": "mean",
    "Temperature": "mean",
    "Wind": "mean"
}).reset_index().rename(columns={
    "Precipitation": "Precipitation_fire",
    "Humidity": "Humidity_fire",
    "Temperature": "Temperature_fire",
    "Wind": "Wind_fire"
})

In [ ]:
# Merge all climate sources into zone summary
df_zone_day = df_zone_fire_counts.merge(df_zone_climate_all, on="Zone_ID", how="left")
df_zone_day = df_zone_day.merge(df_zone_fire_climate, on="Zone_ID", how="left")

In [ ]:
# Use fire-cell climate where fires exist, otherwise zone-wide average
for col in ["Precipitation", "Humidity", "Temperature", "Wind"]:
    df_zone_day[col] = df_zone_day.apply(
        lambda row: row[f"{col}_fire"] if row["Num_Fires"] > 0 and not pd.isna(row[f"{col}_fire"])
        else row[f"{col}_all"],
        axis=1
    )

In [ ]:
# Final zone-day summary
df_zone_day = df_zone_day[["Zone_ID", "Precipitation", "Humidity", "Temperature", "Wind", "Num_Fires"]]
df_zone_day = df_zone_day.sort_values("Zone_ID").reset_index(drop=True)
df_zone_day